OMNIRETAILX – PHASE 5 

CLOUD & SERVERLESS LAYER

In [0]:
from pyspark.sql import functions as F

print("Spark Version:", spark.version)
print("Starting Phase 5 – Cloud Architecture Hardening")

DELTA OPTIMISATION (PRODUCTION STORAGE TUNING)

In [0]:
spark.sql("OPTIMIZE silver_orders")
spark.sql("OPTIMIZE silver_order_items")
spark.sql("OPTIMIZE gold_customer_clv")

print("Delta Optimization Completed")

DATA QUALITY VALIDATION CHECKS

In [0]:
invalid_orders = spark.sql("""
    SELECT COUNT(*) AS invalid_count
    FROM silver_orders
    WHERE order_status IS NULL
""")

duplicate_customers = spark.sql("""
    SELECT customer_id, COUNT(*) c
    FROM silver_customers
    GROUP BY customer_id
    HAVING COUNT(*) > 1
""")

print("Data Quality Results")
invalid_orders.show()
duplicate_customers.show()

SCALABILITY DEMONSTRATION (PARTITIONING)

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE gold_orders_partitioned
    USING DELTA
    PARTITIONED BY (order_date)
    AS SELECT * FROM silver_orders
""")

print("Partitioned Gold Table Created")

GOVERNANCE – UNITY CATALOG PERMISSIONS (SIMULATION)

In [0]:
try:
    spark.sql("GRANT SELECT ON TABLE gold_customer_clv TO `account users`")
    print("Governance Permission Applied")
except:
    print("Permission already exists or insufficient privileges")

MONITORING METRICS (SYSTEM TABLES)

In [0]:
recent_queries = spark.sql("""
    SELECT
        statement_text,
        execution_duration_ms,
        read_rows,
        written_rows
    FROM system.query.history
    ORDER BY start_time DESC
    LIMIT 5
""")

print("Recent Query Monitoring Snapshot")
recent_queries.show(truncate=False)

STORAGE SIZE MONITORING

In [0]:
print("Storage Metrics for Gold Table")
spark.sql("DESCRIBE DETAIL gold_customer_clv").show(truncate=False)

DELTA TIME TRAVEL TEST

In [0]:
spark.sql("""
    CREATE OR REPLACE TABLE gold_customer_clv_backup
    AS SELECT * FROM gold_customer_clv VERSION AS OF 0
""")

print("Delta Time Travel Recovery Validated")

FINAL VALIDATION SUMMARY

In [0]:
total_orders = spark.sql("SELECT COUNT(*) FROM silver_orders")
total_customers = spark.sql("SELECT COUNT(*) FROM silver_customers")
total_stream_records = spark.sql("SELECT COUNT(*) FROM streaming_minute_metrics")

print("Final Metrics:")
total_orders.show()
total_customers.show()
total_stream_records.show()

print("PHASE 5 COMPLETED SUCCESSFULLY")
print("Enterprise Cloud Hardening Applied")

1. Compute vs Storage Separation
    
    Object storage (S3/ADLS) decouples data from compute.

    Compute clusters can scale independently.

    This improves elasticity and cost control.

2. Elastic Scaling
    
    Serverless compute automatically adjusts resources based on workload demand.
    
    Prevents over-provisioning and reduces idle cost.

3. Fault Tolerance
    
    Implemented through:
    - Delta ACID transactions
    - Spark lineage recomputation
    - Checkpoint-based streaming recovery
    - Time travel rollback

4. Security Considerations
    - IAM role-based access control
    - Unity Catalog permissions
    - Encrypted storage
    - No public endpoint exposure
    - VPC isolation

5. Cost Optimization
    - Partition pruning
    - Delta OPTIMIZE
    - Avoiding unnecessary streaming
    - Serverless pay-per-use model